In [ ]:
import kagglehub

kagglehub.login()




In [ ]:
path = kagglehub.dataset_download("fritzstevenson/oregons-historical-wildfires")

In [1]:
import pandas as pd

fires = pd.read_csv('or_historical_wildfires.csv') 
print(fires.shape)
print(fires.columns.tolist())
print(fires.head(3).to_string())
print(fires.dtypes)


(66311, 15)
['id', 'fire_year', 'report_date', 'county', 'latitude', 'longitude', 'total_acres', 'odf_acres', 'fuel_model', 'fuel_descr', 'general_cause', 'fire_name', 'district', 'unit', 'legal']
   id  fire_year report_date      county  latitude  longitude  total_acres  odf_acres fuel_model         fuel_descr   general_cause fire_name        district       unit          legal
0   1       1961  1961-01-25  Washington       NaN        NaN         8.00       8.00          X  non-wildland fuel   Miscellaneous  61511004  51 - Tillamook  Tillamook  T000 R000 S00
1   2       1961  1961-04-04  Washington       NaN        NaN         5.00       5.00          X  non-wildland fuel  Debris Burning  61511009  51 - Tillamook  Tillamook  T000 R000 S00
2   3       1961  1961-04-09  Washington       NaN        NaN         0.05       0.05          X  non-wildland fuel   Recreationist  61511011  51 - Tillamook  Tillamook  T000 R000 S00
id                 int64
fire_year          int64
report_date      

In [4]:
import pandas as pd

fires = pd.read_csv('or_historical_wildfires.csv')  # update filename

total = len(fires)
missing_lat = fires['latitude'].isna().sum()
missing_lon = fires['longitude'].isna().sum()
has_both = fires[['latitude', 'longitude']].dropna().shape[0]

print(f"Total fires:              {total:,}")
print(f"Missing latitude:         {missing_lat:,} ({missing_lat/total*100:.1f}%)")
print(f"Missing longitude:        {missing_lon:,} ({missing_lon/total*100:.1f}%)")
print(f"Has both lat & lon:       {has_both:,} ({has_both/total*100:.1f}%)")
print(f"\nCounty missing values:    {fires['county'].isna().sum():,}")
print(f"Unique counties:          {fires['county'].nunique()}")
print(f"\nFires in 1970-2021 range: {fires[fires['fire_year'].between(1970,2021)].shape[0]:,}")

Total fires:              66,311
Missing latitude:         7,197 (10.9%)
Missing longitude:        7,196 (10.9%)
Has both lat & lon:       59,114 (89.1%)

County missing values:    16
Unique counties:          36

Fires in 1970-2021 range: 56,346


In [7]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

# ── Load data ─────────────────────────────────────────────────────────
fires = pd.read_csv('or_historical_wildfires.csv')
weather = pd.read_csv('oregon_weather_cleaned.csv', parse_dates=['DATE'])

# ── Prep fires ────────────────────────────────────────────────────────
fires['report_date'] = pd.to_datetime(fires['report_date'], errors='coerce')
fires = fires[fires['fire_year'].between(1970, 2021)].copy()
fires = fires.dropna(subset=['report_date'])
print(f"Fires ready to merge: {len(fires):,}")

# ── Prep weather: one row per station per day ─────────────────────────
weather_cols = ['DATE','STATION','NAME','LATITUDE','LONGITUDE',
                'TEMP','MAX','MIN','DEWP','WDSP','MXSPD',
                'PRCP','PRCP_30DAY','VPD_PROXY','TEMP_RANGE','FIRE_SEASON']
wdf = weather[weather_cols].dropna(subset=['LATITUDE','LONGITUDE']).copy()

# ── Get unique station locations for spatial index ────────────────────
stations = wdf[['STATION','NAME','LATITUDE','LONGITUDE']].drop_duplicates('STATION')
station_coords = stations[['LATITUDE','LONGITUDE']].values
tree = cKDTree(station_coords)

# ── For each fire with lat/lon, find nearest weather station ──────────
fires_with_loc = fires.dropna(subset=['latitude','longitude']).copy()
fire_coords = fires_with_loc[['latitude','longitude']].values
_, idx = tree.query(fire_coords)
fires_with_loc['NEAREST_STATION'] = stations.iloc[idx]['STATION'].values
fires_with_loc['NEAREST_STATION_NAME'] = stations.iloc[idx]['NAME'].values

print(f"Fires matched to nearest station: {len(fires_with_loc):,}")

# ── Merge on station + date ───────────────────────────────────────────
merged = fires_with_loc.merge(
    wdf,
    left_on  = ['report_date', 'NEAREST_STATION'],
    right_on = ['DATE', 'STATION'],
    how='left'
)

print(f"After merge shape: {merged.shape}")
print(f"Weather match rate: {merged['TEMP'].notna().sum()/len(merged)*100:.1f}%")
print(merged[['fire_name','report_date','county','total_acres',
              'TEMP','MAX','PRCP','WDSP','VPD_PROXY','PRCP_30DAY']].head(10).to_string())

# # ── Save ──────────────────────────────────────────────────────────────
# merged.to_csv('oregon_fires_weather_merged.csv', index=False)
# print("\n Saved to oregon_fires_weather_merged.csv!")

Fires ready to merge: 56,346
Fires matched to nearest station: 56,280
After merge shape: (56280, 33)
Weather match rate: 40.5%
  fire_name report_date     county  total_acres  TEMP  MAX  PRCP  WDSP  VPD_PROXY  PRCP_30DAY
0  70511204  1970-06-15  Tillamook         0.05   NaN  NaN   NaN   NaN        NaN         NaN
1  70511209  1970-06-22  Tillamook         0.75   NaN  NaN   NaN   NaN        NaN         NaN
2  70511213  1970-06-24  Tillamook        10.00   NaN  NaN   NaN   NaN        NaN         NaN
3  70511231  1970-07-04  Tillamook         0.05   NaN  NaN   NaN   NaN        NaN         NaN
4  70511237  1970-07-06  Tillamook         0.05   NaN  NaN   NaN   NaN        NaN         NaN
5  70511245  1970-07-08  Tillamook         0.05   NaN  NaN   NaN   NaN        NaN         NaN
6  70511254  1970-07-11  Tillamook         0.75   NaN  NaN   NaN   NaN        NaN         NaN
7  70511257  1970-07-12  Tillamook         0.05   NaN  NaN   NaN   NaN        NaN         NaN
8  70511267  1970-07-14  Ti

Weather match rate is too low. Try rolling average instead of direct match: 

In [9]:

# ── Load data ─────────────────────────────────────────────────────────
fires = pd.read_csv('or_historical_wildfires.csv')
weather = pd.read_csv('oregon_weather_cleaned.csv', parse_dates=['DATE'])

# ── Interpolate missing weather values per station ────────────────────
weather_cols = ['TEMP','MAX','MIN','DEWP','WDSP','MXSPD',
                'PRCP','PRCP_30DAY','VPD_PROXY','TEMP_RANGE']

print("Filling missing values with 7-day rolling average per station...")
weather = weather.sort_values(['STATION', 'DATE'])

for col in weather_cols:
    # Compute 7-day rolling mean per station (excluding the NaN itself)
    rolling = (weather.groupby('STATION')[col]
                      .transform(lambda x: x.fillna(
                          x.rolling(7, min_periods=1, center=True).mean()
                      )))
    weather[col] = rolling

# Check improvement
print(f"TEMP null rate after fill: {weather['TEMP'].isna().sum()/len(weather)*100:.1f}%")
print(f"PRCP null rate after fill: {weather['PRCP'].isna().sum()/len(weather)*100:.1f}%")

# ── Prep fires ────────────────────────────────────────────────────────
fires['report_date'] = pd.to_datetime(fires['report_date'], errors='coerce')
fires = fires[fires['fire_year'].between(1970, 2021)].copy()
fires = fires.dropna(subset=['report_date', 'latitude', 'longitude'])
print(f"\nFires ready: {len(fires):,}")

# ── Build spatial index ───────────────────────────────────────────────
wdf = weather.dropna(subset=['LATITUDE','LONGITUDE']).copy()
stations = wdf[['STATION','LATITUDE','LONGITUDE']].drop_duplicates('STATION')
tree = cKDTree(stations[['LATITUDE','LONGITUDE']].values)

# ── Find nearest station per fire ─────────────────────────────────────
fire_coords = fires[['latitude','longitude']].values
_, idx = tree.query(fire_coords)
fires['NEAREST_STATION'] = stations.iloc[idx]['STATION'].values

# ── Merge on station + date ───────────────────────────────────────────
merge_cols = ['DATE','STATION'] + weather_cols + ['FIRE_SEASON','MONTH','DOY']
merged = fires.merge(
    wdf[merge_cols],
    left_on  = ['report_date', 'NEAREST_STATION'],
    right_on = ['DATE', 'STATION'],
    how='left'
)

match_rate = merged['TEMP'].notna().sum() / len(merged) * 100
print(f"Final shape: {merged.shape}")
print(f"Weather match rate: {match_rate:.1f}%")
print(merged[['fire_name','report_date','county','total_acres',
              'TEMP','MAX','PRCP','WDSP','VPD_PROXY']].head(10).to_string())

merged.to_csv('oregon_fires_weather_merged.csv', index=False)
print("\n Saved to oregon_fires_weather_merged.csv!")

Filling missing values with 7-day rolling average per station...
TEMP null rate after fill: 0.0%
PRCP null rate after fill: 0.7%

Fires ready: 56,280
Final shape: (56280, 31)
Weather match rate: 40.5%
  fire_name report_date     county  total_acres  TEMP  MAX  PRCP  WDSP  VPD_PROXY
0  70511204  1970-06-15  Tillamook         0.05   NaN  NaN   NaN   NaN        NaN
1  70511209  1970-06-22  Tillamook         0.75   NaN  NaN   NaN   NaN        NaN
2  70511213  1970-06-24  Tillamook        10.00   NaN  NaN   NaN   NaN        NaN
3  70511231  1970-07-04  Tillamook         0.05   NaN  NaN   NaN   NaN        NaN
4  70511237  1970-07-06  Tillamook         0.05   NaN  NaN   NaN   NaN        NaN
5  70511245  1970-07-08  Tillamook         0.05   NaN  NaN   NaN   NaN        NaN
6  70511254  1970-07-11  Tillamook         0.75   NaN  NaN   NaN   NaN        NaN
7  70511257  1970-07-12  Tillamook         0.05   NaN  NaN   NaN   NaN        NaN
8  70511267  1970-07-14  Tillamook         0.05   NaN  NaN   

In [10]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

fires = pd.read_csv('or_historical_wildfires.csv')
weather = pd.read_csv('oregon_weather_cleaned.csv', parse_dates=['DATE'])

fires['report_date'] = pd.to_datetime(fires['report_date'], errors='coerce')
fires = fires[fires['fire_year'].between(1970, 2021)].copy()
fires = fires.dropna(subset=['report_date', 'latitude', 'longitude'])

# ── Check what dates the weather data actually covers ─────────────────
print("Weather date range:", weather['DATE'].min(), "to", weather['DATE'].max())
print("Weather unique years:", sorted(weather['DATE'].dt.year.unique())[:10], "...")
print(f"Total weather rows: {len(weather):,}")

# ── Check nearest station for first few fires ─────────────────────────
stations = weather[['STATION','LATITUDE','LONGITUDE']].drop_duplicates('STATION').dropna()
tree = cKDTree(stations[['LATITUDE','LONGITUDE']].values)

sample_fires = fires.head(10)
fire_coords = sample_fires[['latitude','longitude']].values
_, idx = tree.query(fire_coords)
sample_fires = sample_fires.copy()
sample_fires['NEAREST_STATION'] = stations.iloc[idx]['STATION'].values

# For each fire, check if that station has data around that date
for _, row in sample_fires.iterrows():
    station = row['NEAREST_STATION']
    date = row['report_date']
    station_data = weather[weather['STATION'] == station]
    nearby = station_data[abs((station_data['DATE'] - date).dt.days) <= 7]
    print(f"Fire {row['fire_name']} on {date.date()} → Station {station} "
          f"| Station date range: {station_data['DATE'].min().date()} - {station_data['DATE'].max().date()} "
          f"| Nearby records: {len(nearby)}")
    

Weather date range: 1970-01-01 00:00:00 to 2021-12-31 00:00:00
Weather unique years: [np.int32(1970), np.int32(1971), np.int32(1972), np.int32(1973), np.int32(1974), np.int32(1975), np.int32(1976), np.int32(1977), np.int32(1978), np.int32(1979)] ...
Total weather rows: 509,866
Fire 70511204 on 1970-06-15 → Station 72698799999 | Station date range: 1973-01-01 - 2005-12-19 | Nearby records: 0
Fire 70511209 on 1970-06-22 → Station 72698799999 | Station date range: 1973-01-01 - 2005-12-19 | Nearby records: 0
Fire 70511213 on 1970-06-24 → Station 72698794204 | Station date range: 2006-03-03 - 2013-09-30 | Nearby records: 0
Fire 70511231 on 1970-07-04 → Station 72698794204 | Station date range: 2006-03-03 - 2013-09-30 | Nearby records: 0
Fire 70511237 on 1970-07-06 → Station 72698524242 | Station date range: 2006-01-01 - 2021-12-31 | Nearby records: 0
Fire 70511245 on 1970-07-08 → Station 72698799999 | Station date range: 1973-01-01 - 2005-12-19 | Nearby records: 0
Fire 70511254 on 1970-07-1

Some early stations don't have data going back to 1970s. Instead, for each fire find the closest valid station 

In [11]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

# ── Load data ─────────────────────────────────────────────────────────
fires = pd.read_csv('or_historical_wildfires.csv')
weather = pd.read_csv('oregon_weather_cleaned.csv', parse_dates=['DATE'])

# ── Rolling average fill ──────────────────────────────────────────────
weather_cols = ['TEMP','MAX','MIN','DEWP','WDSP','MXSPD',
                'PRCP','PRCP_30DAY','VPD_PROXY','TEMP_RANGE']
weather = weather.sort_values(['STATION', 'DATE'])
for col in weather_cols:
    weather[col] = weather.groupby('STATION')[col].transform(
        lambda x: x.fillna(x.rolling(7, min_periods=1, center=True).mean())
    )

# ── Prep fires ────────────────────────────────────────────────────────
fires['report_date'] = pd.to_datetime(fires['report_date'], errors='coerce')
fires = fires[fires['fire_year'].between(1970, 2021)].copy()
fires = fires.dropna(subset=['report_date', 'latitude', 'longitude'])
print(f"Fires ready: {len(fires):,}")

# ── Build weather lookup indexed by station + date ────────────────────
merge_cols = ['DATE','STATION','LATITUDE','LONGITUDE'] + weather_cols + ['FIRE_SEASON','MONTH','DOY']
wdf = weather[merge_cols].dropna(subset=['LATITUDE','LONGITUDE']).copy()
weather_lookup = wdf.set_index(['STATION','DATE'])

# ── Get unique stations ───────────────────────────────────────────────
stations = wdf[['STATION','LATITUDE','LONGITUDE']].drop_duplicates('STATION').reset_index(drop=True)

# ── Build spatial index with k=10 nearest stations ───────────────────
tree = cKDTree(stations[['LATITUDE','LONGITUDE']].values)
fire_coords = fires[['latitude','longitude']].values
_, indices = tree.query(fire_coords, k=10)  # get 10 nearest candidates

# ── Match each fire to nearest station that has data on fire date ─────
print("Matching fires to stations with active data on fire date...")
results = []

for i, (_, row) in enumerate(fires.iterrows()):
    date = row['report_date']
    matched_weather = {}

    # Try each of the 10 nearest stations
    for j in range(10):
        station_id = stations.iloc[indices[i, j]]['STATION']
        key = (station_id, date)
        if key in weather_lookup.index:
            w = weather_lookup.loc[key]
            matched_weather = w.to_dict() if isinstance(w, pd.Series) else w.iloc[0].to_dict()
            matched_weather['STATION_USED'] = station_id
            break

    results.append({**row.to_dict(), **matched_weather})

    if (i + 1) % 5000 == 0:
        print(f"  Processed {i+1:,}/{len(fires):,}...")

# ── Assemble final dataframe ──────────────────────────────────────────
merged = pd.DataFrame(results)
match_rate = merged['TEMP'].notna().sum() / len(merged) * 100
print(f"\nFinal shape: {merged.shape}")
print(f"Weather match rate: {match_rate:.1f}%")
print(merged[['fire_name','report_date','county','total_acres',
              'TEMP','MAX','PRCP','WDSP','VPD_PROXY']].head(10).to_string())

merged.to_csv('oregon_fires_weather_merged.csv', index=False)
print("\n Saved to oregon_fires_weather_merged.csv!")

Fires ready: 56,280
Matching fires to stations with active data on fire date...
  Processed 5,000/56,280...
  Processed 10,000/56,280...
  Processed 15,000/56,280...
  Processed 20,000/56,280...
  Processed 25,000/56,280...
  Processed 30,000/56,280...
  Processed 35,000/56,280...
  Processed 40,000/56,280...
  Processed 45,000/56,280...
  Processed 50,000/56,280...
  Processed 55,000/56,280...

Final shape: (56280, 31)
Weather match rate: 99.6%
  fire_name report_date     county  total_acres  TEMP    MAX  PRCP  WDSP  VPD_PROXY
0  70511204  1970-06-15  Tillamook         0.05  59.2   64.9   0.0   6.9        8.4
1  70511209  1970-06-22  Tillamook         0.75  69.6   84.0   0.0   7.6       18.3
2  70511213  1970-06-24  Tillamook        10.00  68.2   84.9   0.0   7.1       18.6
3  70511231  1970-07-04  Tillamook         0.05  76.2   99.0   0.0   7.0       19.2
4  70511237  1970-07-06  Tillamook         0.05  64.5   82.0   0.0   7.7       15.3
5  70511245  1970-07-08  Tillamook         0.0